In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree


class NDealkylation(MorphingOperator):
    def __init__(self):
        super(NDealkylation, self).__init__()
        self._name = "N-Dealkylation (Phase I - Advanced)"
        self._target_bonds = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;H0,H1;!a;!$(N-C=O)][CX4;H1,H2,H3]")

    def setOriginal(self, mol):
        super(NDealkylation, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_bonds.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        
        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
            
        idx_n, idx_c = random.choice(self._target_bonds)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx_n, idx_c)
            if bond:
                rw_mol.RemoveBond(idx_n, idx_c)
                
            new_mol = rw_mol.GetMol()
            
            atom_n = new_mol.GetAtomWithIdx(idx_n)
            atom_n.SetNoImplicit(False)
            atom_n.SetNumExplicitHs(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            frags_mols = Chem.GetMolFrags(new_mol, asMols=True)
            
            if frags_mols:
                
                frags_indices = Chem.GetMolFrags(new_mol, asMols=False)
                frags_with_meta = list(zip(frags_mols, frags_indices))
                frags_with_meta = sorted(
                    frags_with_meta, 
                    key=lambda x: (idx_n in x[1], x[0].GetNumAtoms()), 
                    reverse=True
                )
                
                final_mol = frags_with_meta[0][0]
                
                Chem.AssignStereochemistry(final_mol, cleanIt=True, force=True)
                clean_smiles = Chem.MolToSmiles(final_mol)
                return MolpherMol(clean_smiles)
                
            return MolpherMol(other=rdkit_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

dealk_op = NDealkylation()

test_dealk_molecules = {
    "1. Μεθυλαμινο-προπάνιο (Βασικό τεστ)": "CCCNC",
    
    "2. Παγίδα Μεγέθους (Το fragment που φεύγει είναι μεγαλύτερο)": "CNCCCCC", 
    # Αν κοπεί η μεγάλη αλυσίδα, το fragment που μένει (CN) είναι μικρότερο από αυτό που φεύγει.
    
    "3. Παγίδα Αζώτου και στα 2 κομμάτια (Διμεθυλαμινο-αιθυλαμίνη)": "CCN(C)CCN",
    # Αν κοπεί ο δεσμός, και τα δύο κομμάτια έχουν άζωτο.
    
    "4. Παγίδα Ίσου Μεγέθους με Άζωτο (Αιθυλο-μεθυλαμίνη)": "CCN(C)CC",
    
    "5. Σύνθετο Φάρμακο (Ιμιπραμίνη - Αντικαταθλιπτικό)": "CN(C)CCCN1c2ccccc2CCc3ccccc13",
    # Δύο διαφορετικά είδη αζώτων (αλυσίδας και τρικυκλικό).
    
    "6. Παγίδα Δακτυλίου (N-methylpiperidine -> Άνοιγμα δακτυλίου αντί για fragment)": "CN1CCCCC1",
    # Αν ο αλγόριθμος επιλέξει να κόψει δεσμό ΜΕΣΑ στον δακτύλιο, παίρνουμε 1 ενιαίο fragment 
    # (γραμμικό) αντί για δύο ξεχωριστά.
    
    "7. Καφεΐνη (Σύνθετο - Αρωματικά/Αμιδικά Άζωτα -> Πρέπει να αγνοηθούν!)": "CN1C=NC2=C1C(=O)N(C)C(=O)N2C",
    
    "8. Μεθυλακεταμίδιο (Αμίδιο -> Πρέπει να αγνοηθεί)": "CNC(C)=O"
}


print("=== STARTING N-DEALKYLATION TESTING ===")
for name, smiles in test_dealk_molecules.items():
    mol = MolpherMol(smiles)
    dealk_op.setOriginal(mol)
    product = dealk_op.morph()
    
    print(f"\n{name}")
    print(f"  SOURCE: {mol.getSMILES()}")
    print(f"  TARGET: {product.getSMILES() if product and product.getSMILES() != mol.getSMILES() else 'No change (Safe)'}")
print("\n=======================================")

=== STARTING N-DEALKYLATION TESTING ===

1. Μεθυλαμινο-προπάνιο (Βασικό τεστ)
  SOURCE: CCCNC
  TARGET: CN

2. Παγίδα Μεγέθους (Το fragment που φεύγει είναι μεγαλύτερο)
  SOURCE: CCCCCNC
  TARGET: CN

3. Παγίδα Αζώτου και στα 2 κομμάτια (Διμεθυλαμινο-αιθυλαμίνη)
  SOURCE: CCN(C)CCN
  TARGET: CCNC

4. Παγίδα Ίσου Μεγέθους με Άζωτο (Αιθυλο-μεθυλαμίνη)
  SOURCE: CCN(C)CC
  TARGET: CCNCC

5. Σύνθετο Φάρμακο (Ιμιπραμίνη - Αντικαταθλιπτικό)
  SOURCE: CN(C)CCCN1C2=CC=CC=C2CCC2=CC=CC=C21
  TARGET: CNCCCN1C2=CC=CC=C2CCC2=CC=CC=C21

6. Παγίδα Δακτυλίου (N-methylpiperidine -> Άνοιγμα δακτυλίου αντί για fragment)
  SOURCE: CN1CCCCC1
  TARGET: CCCCCNC

7. Καφεΐνη (Σύνθετο - Αρωματικά/Αμιδικά Άζωτα -> Πρέπει να αγνοηθούν!)
  SOURCE: CN1C=NC2=C1C(=O)N(C)C(=O)N2C
  TARGET: No change (Safe)

8. Μεθυλακεταμίδιο (Αμίδιο -> Πρέπει να αγνοηθεί)
  SOURCE: CNC(C)=O
  TARGET: No change (Safe)

